# Data Preparation — Train / Val / Test Split

This notebook runs **once** and produces the shared split files used by all approaches.

**Output files (saved to `final_data/`):**
- `train.jsonl` — 80% of data
- `val.jsonl` — 10% of data
- `test.jsonl` — 10% of data
- `mlb.joblib` — fitted `MultiLabelBinarizer` (same label encoding for all approaches)

**Why shuffle before split?**  
The original `final1.jsonl` is ordered by category — all samples of one intent appear together before the next intent begins. Without shuffling, iterative stratification would still produce correct label proportions but the train/val/test sets would contain contiguous blocks from the original file, potentially picking up any conversation-order biases.

**Why iterative stratification?**  
Standard `train_test_split` does not preserve multi-label distributions. `IterativeStratification` from `scikit-multilearn` ensures each split has a proportional representation of all 32 intent classes, including rare ones like `ask_find_product` (47 samples).

In [ ]:
!pip install scikit-multilearn scikit-learn numpy pandas joblib --quiet

In [ ]:
import json
import numpy as np
import pandas as pd
import joblib
from collections import Counter
from sklearn.preprocessing import MultiLabelBinarizer
from skmultilearn.model_selection import iterative_train_test_split

DATA_PATH   = '../final_data/final1.jsonl'
OUT_DIR     = '../final_data'
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

## 1. Load Data

In [ ]:
texts, labels = [], []

with open(DATA_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        item = json.loads(line)
        texts.append(item['text'])
        labels.append(item['cats'])

print(f"Total samples    : {len(texts)}")
print(f"Multi-label rows : {sum(1 for l in labels if len(l) > 1)}")
print(f"Single-label rows: {sum(1 for l in labels if len(l) == 1)}")
print()
print("First 5 labels (before shuffle):")
for i in range(5):
    print(f"  [{i}] {labels[i]}")

## 2. Shuffle Data

The original file is ordered by category. Shuffling with a fixed seed ensures:
- Reproducibility: the same shuffle is produced every run
- No order bias entering the train/val/test splits

In [ ]:
indices = np.random.permutation(len(texts))
texts  = [texts[i]  for i in indices]
labels = [labels[i] for i in indices]

print(f"Shuffled {len(texts)} samples (seed={RANDOM_SEED}).")
print()
print("First 5 labels (after shuffle):")
for i in range(5):
    print(f"  [{i}] {labels[i]}")

## 3. Fit MultiLabelBinarizer

Fit once here and save to disk. All approaches load this same instance to guarantee identical label ordering.

In [ ]:
mlb = MultiLabelBinarizer()
Y = mlb.fit_transform(labels)

print(f"Label matrix shape : {Y.shape}")
print(f"Classes ({len(mlb.classes_)}): {list(mlb.classes_)}")

## 4. Iterative Stratification Split (80 / 10 / 10)

In [ ]:
X_arr = np.array(texts).reshape(-1, 1)  # skmultilearn requires 2D

# 80% train, 20% temp
X_train_arr, y_train, X_temp_arr, y_temp = iterative_train_test_split(
    X_arr, Y, test_size=0.2
)

# Split temp 50/50 → 10% val, 10% test
X_val_arr, y_val, X_test_arr, y_test = iterative_train_test_split(
    X_temp_arr, y_temp, test_size=0.5
)

X_train = X_train_arr.flatten().tolist()
X_val   = X_val_arr.flatten().tolist()
X_test  = X_test_arr.flatten().tolist()

# Convert binary matrix back to label lists for JSONL storage
labels_train = [list(l) for l in mlb.inverse_transform(y_train)]
labels_val   = [list(l) for l in mlb.inverse_transform(y_val)]
labels_test  = [list(l) for l in mlb.inverse_transform(y_test)]

print(f"Train : {len(X_train):4d} samples ({len(X_train)/len(texts)*100:.1f}%)")
print(f"Val   : {len(X_val):4d} samples ({len(X_val)/len(texts)*100:.1f}%)")
print(f"Test  : {len(X_test):4d} samples ({len(X_test)/len(texts)*100:.1f}%)")

## 5. Verify Label Distribution

Check that all 32 labels are present in every split, especially rare ones.

In [ ]:
def label_freq(label_lists):
    return Counter(lbl for sample in label_lists for lbl in sample)

freq_all   = label_freq(labels)
freq_train = label_freq(labels_train)
freq_val   = label_freq(labels_val)
freq_test  = label_freq(labels_test)

df_check = pd.DataFrame({
    'Total' : freq_all,
    'Train' : freq_train,
    'Val'   : freq_val,
    'Test'  : freq_test,
}).fillna(0).astype(int).sort_values('Total', ascending=False)

print(df_check.to_string())

missing = [cls for cls in mlb.classes_ if freq_test.get(cls, 0) == 0]
if missing:
    print(f"\nWARNING — labels missing from test set: {missing}")
else:
    print("\nAll labels present in all splits.")

## 6. Save Files

In [ ]:
def save_jsonl(texts, label_lists, path):
    with open(path, 'w', encoding='utf-8') as f:
        for text, cats in zip(texts, label_lists):
            json.dump({'text': text, 'cats': cats}, f, ensure_ascii=False)
            f.write('\n')

save_jsonl(X_train, labels_train, f'{OUT_DIR}/train.jsonl')
save_jsonl(X_val,   labels_val,   f'{OUT_DIR}/val.jsonl')
save_jsonl(X_test,  labels_test,  f'{OUT_DIR}/test.jsonl')
joblib.dump(mlb, f'{OUT_DIR}/mlb.joblib')

print("Saved:")
print(f"  {OUT_DIR}/train.jsonl  ({len(X_train)} samples)")
print(f"  {OUT_DIR}/val.jsonl    ({len(X_val)} samples)")
print(f"  {OUT_DIR}/test.jsonl   ({len(X_test)} samples)")
print(f"  {OUT_DIR}/mlb.joblib   ({len(mlb.classes_)} classes)")